# Evaluate ai.summarize(...) Quality with PySpark

This notebook evaluates summaries across five complementary quality dimensions. The AI transformations and LLM-as-a-Judge evaluation remain in Spark. Only the small, materialized result set is converted to pandas for metrics and charts.

### What You'll Do
1. Summarize sample articles with `ai.summarize`.
2. Score fluency, coherence, conciseness, consistency, and relevance.
3. Inspect average and per-sample quality.
4. Summarize complete multi-column support-ticket rows.
5. Compare default summaries with a concise instruction-based variant.

### Before You Start
- **Runtime** - This notebook was made for **Fabric 1.3 runtime**.
- **Customize it** - Replace the sample data and adapt the judge criteria to your use case.
- **Keep comparisons fair** - Hold the judge model and prompts fixed while changing one executor setting at a time.
- **Validate important decisions** - LLM judge scores are useful proxies, not a substitute for human-reviewed production samples.

| Metric | Measures |
|--------|----------|
| **Fluency** | The summary is clear and grammatically natural |
| **Coherence** | Ideas are logically organized |
| **Conciseness** | The summary avoids unnecessary detail |
| **Consistency** | Claims are supported by the source |
| **Relevance** | The most important information is retained |

[ai.summarize PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/summarize)


## 1. Setup

Install Pydantic for the structured response schemas used by the judge.
The baseline uses `gpt-5-mini` with low reasoning effort. A fixed
`gpt-5.1` judge scores each metric independently. Shared verbosity remains unset.


In [ ]:
%pip install -q pydantic 2>/dev/null


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import synapse.ml.spark.aifunc as aifunc
from pydantic import BaseModel, Field
from pyspark.sql import functions as F

# Use the smaller model for the function under test and a larger fixed model for judging.
EXECUTOR_OPTIONS = {
    "deploymentName": "gpt-5-mini",
    "reasoningEffort": "low",
}
JUDGE_OPTIONS = {
    "deploymentName": "gpt-5.1",
    "reasoningEffort": "medium",
}

class MetricEval(BaseModel):
    reason: str = Field(description="Brief rationale for the score")
    score: int = Field(ge=1, le=5, description="Integer score from 1 to 5")

def materialize(frame):
    cached = frame.cache()
    _ = cached.count()
    error_columns = [name for name in cached.columns if name.endswith("_error")]
    if error_columns:
        has_error = F.lit(False)
        for name in error_columns:
            has_error = has_error | F.coalesce(
                F.length(F.trim(F.col(name).cast("string"))) > 0,
                F.lit(False),
            )
        failed_rows = cached.filter(has_error)
        failure_count = failed_rows.count()
        if failure_count:
            print(f"{failure_count} row(s) contain AI Function errors:")
            id_columns = [
                name for name in ("sample_id", "ticket_id") if name in cached.columns
            ]
            display(failed_rows.select(*id_columns, *error_columns))
    return cached

def fresh_ai_view(frame):
    return frame.select("*")

def add_judge_metric(frame, metric_name, prompt, column_prefix=""):
    score_col = f"{column_prefix}{metric_name}"
    response_col = f"_{score_col}_response"
    raw_score_col = f"_{score_col}_raw_score"
    error_col = f"_{score_col}_error"
    judged = fresh_ai_view(frame).ai.generate_response(
        prompt=prompt,
        is_prompt_template=True,
        output_col=response_col,
        error_col=error_col,
        response_format=MetricEval,
        **JUDGE_OPTIONS,
    )
    invalid_score = (
        F.col(raw_score_col).isNull()
        | ~F.col(raw_score_col).between(1, 5)
        | (F.col(raw_score_col) != F.floor(F.col(raw_score_col)))
    )
    validation_message = "Judge score must be an integer from 1 to 5"
    return (
        judged
        .withColumn(
            raw_score_col,
            F.get_json_object(F.col(response_col), "$.score").cast("double"),
        )
        .withColumn(
            error_col,
            F.when(
                invalid_score,
                F.when(
                    F.length(
                        F.trim(F.coalesce(F.col(error_col), F.lit("")))
                    ) > 0,
                    F.concat(
                        F.col(error_col),
                        F.lit(f"; {validation_message}"),
                    ),
                ).otherwise(F.lit(validation_message)),
            ).otherwise(F.col(error_col)),
        )
        .withColumn(
            score_col,
            F.when(
                ~invalid_score,
                F.col(raw_score_col).cast("int"),
            ),
        )
        .withColumn(
            f"{score_col}_reason",
            F.get_json_object(F.col(response_col), "$.reason"),
        )
        .drop(raw_score_col)
    )


## 2. Load Sample Data


In [ ]:
rows = [(1,
  'The Federal Reserve held interest rates steady at its January meeting, keeping the benchmark federal '
  'funds rate in the 5.25% to 5.50% range for the fourth consecutive session. Fed Chair Jerome Powell said '
  "during the post-meeting press conference that while inflation has moved closer to the central bank's 2% "
  'target, policymakers want to see "more evidence" before beginning to cut rates. The consumer price index '
  "rose 3.1% year-over-year in December, down from a peak of 9.1% in June 2022 but still above the Fed's "
  'goal. Labor markets remain resilient, with the economy adding 216,000 jobs in December and the '
  'unemployment rate holding at 3.7%. Powell emphasized that the committee does not expect it will be '
  'appropriate to reduce rates until it has "greater confidence that inflation is moving sustainably toward '
  '2 percent." Markets had priced in a roughly 50% chance of a rate cut by March, but Powell\'s comments '
  'pushed those expectations out to May or June. Treasury yields ticked higher following the announcement, '
  'with the 10-year note rising 5 basis points to 4.07%. Stock futures initially dipped on the news but '
  'recovered by end of trading as investors digested the overall dovish tone of the statement, which removed '
  'language about potential further tightening.'),
 (2,
  'Researchers at Stanford University and the Allen Institute for AI have published a landmark study '
  'demonstrating that large language models trained on synthetic data can match or exceed the performance of '
  'models trained on human-curated datasets across a range of natural language processing benchmarks. The '
  'study, published in Nature Machine Intelligence, evaluated a family of models called SynthLM ranging from '
  '1.3 billion to 70 billion parameters. The researchers generated training data by prompting an existing '
  'frontier model to produce question-answer pairs, reasoning chains, and summarization examples, then '
  'filtered outputs for quality using a combination of automated checks and a small set of human reviewers. '
  'On the MMLU benchmark, the 70B SynthLM model scored 84.2%, compared to 83.7% for a model of the same size '
  'trained on the Pile dataset. On summarization tasks using the CNN/DailyMail dataset, SynthLM achieved a '
  'ROUGE-L score of 42.8 versus 41.5 for the baseline. The researchers noted that synthetic data generation '
  'cost approximately $2.3 million, compared to an estimated $15 million for curating an equivalent volume '
  'of human-labeled data. However, Dr. Maria Chen, the lead author, cautioned that "synthetic data amplifies '
  'any biases present in the source model" and recommended hybrid approaches that combine synthetic and '
  'human-curated data for safety-critical applications. The team has released SynthLM-7B under an open '
  'license for further research.'),
 (3,
  'Bristol-Myers Squibb announced positive results from its Phase III clinical trial of mavacamten in '
  'patients with obstructive hypertrophic cardiomyopathy, a condition affecting roughly 1 in 500 people in '
  'which the heart muscle becomes abnormally thick and can obstruct blood flow. The VALOR-HCM trial enrolled '
  '112 patients across 34 clinical sites in the United States and Europe who were already being considered '
  'for septal reduction therapy - an invasive procedure to thin the heart muscle. After 16 weeks of '
  'treatment, only 18% of patients in the mavacamten group still met the guideline criteria for the surgical '
  'procedure, compared to 77% in the placebo group. Patients receiving mavacamten also showed significant '
  'improvements in exercise capacity, with a mean increase of 2.8 mL/kg/min in peak oxygen consumption '
  'compared to 0.3 mL/kg/min in the placebo arm. Quality of life scores measured by the Kansas City '
  'Cardiomyopathy Questionnaire improved by an average of 9.1 points in the treatment group versus 1.8 '
  'points for placebo. Dr. Jonathan Ho, the principal investigator at Massachusetts General Hospital, stated '
  'that "these results confirm mavacamten\'s potential to fundamentally change how we treat patients with '
  'obstructive HCM, offering a non-invasive alternative to surgery." Common side effects included dizziness '
  '(15%), fatigue (12%), and atrial fibrillation (6%). Bristol-Myers Squibb plans to submit the data to the '
  'FDA for label expansion by mid-2025.'),
 (4,
  'The city of Austin, Texas approved a $7.1 billion transit expansion plan on Tuesday that will bring two '
  'new light rail lines, 30 miles of dedicated bus rapid transit lanes, and a downtown tunnel connecting the '
  "city's east and west corridors. The plan, known as Project Connect Phase 2, passed with 58% voter "
  'approval after a contentious campaign that pitted transit advocates against opponents who argued the '
  "costs would strain the city's budget. Construction on the first light rail segment - a 12.4-mile Orange "
  'Line running from the Austin-Bergstrom International Airport through downtown to the North Lamar Transit '
  'Center - is expected to begin in 2026 with service starting in 2031. The second Blue Line will run '
  'east-west from the growing Mueller neighborhood through the University of Texas campus to the Westgate '
  'shopping district. Capital Metro CEO Randy Clarke said the system is projected to carry 87,000 daily '
  'riders by 2040, reducing car trips on the I-35 corridor by an estimated 14%. The project will be funded '
  'through a combination of federal grants (35%), a property tax increase of 8.75 cents per $100 of assessed '
  'valuation (40%), and revenue bonds (25%). Critics, including the Austin Taxpayers Association, have '
  'argued that the cost-per-mile of $275 million for the light rail exceeds comparable projects in Denver '
  "and Portland and that ridership projections are overly optimistic given the city's sprawling geography."),
 (5,
  'Patagonia announced a sweeping overhaul of its supply chain on Wednesday, committing to sourcing 100% of '
  'its cotton from regenerative organic farms by 2030 and transitioning all polyester products to recycled '
  'or bio-based materials by 2028. The outdoor apparel company said it will invest $340 million over five '
  'years to support the transition, including $120 million in direct grants to farming cooperatives in '
  'India, Peru, and the United States that adopt regenerative practices such as cover cropping, reduced '
  'tillage, and integrated pest management. Patagonia currently sources about 34% of its cotton from '
  'regenerative farms, up from less than 2% in 2018. CEO Ryan Gellert said in a statement that "the climate '
  'crisis requires us to move faster and invest more heavily in the solutions we know work. Regenerative '
  'agriculture is not just about reducing harm - it actively restores soil health and sequesters carbon." An '
  'independent lifecycle analysis conducted by the consulting firm Quantis estimated that full adoption of '
  "regenerative cotton would reduce Patagonia's Scope 3 emissions by approximately 18%, or 46,000 metric "
  'tons of CO2 equivalent per year. The company also announced it will publish a quarterly supply chain '
  'transparency report starting in Q1 2025, disclosing factory-level audit results, worker wage data, and '
  'environmental impact metrics for each of its 72 Tier 1 suppliers.')]
df = spark.createDataFrame(rows, ["sample_id", "article"])
display(df)


## 3. Run `ai.summarize`


In [ ]:
summary_df = materialize(
    df.ai.summarize(
        input_col="article",
        output_col="summary",
        error_col="executor_error",
        **EXECUTOR_OPTIONS,
    )
)
display(summary_df.select("article", "summary"))
display(summary_df.ai.stats)


## 4. Evaluate with an LLM Judge


In [ ]:
EVAL_METRICS = {
    "fluency": """Score fluency from 1 to 5.
A score of 5 means the summary is clear, grammatical, and natural to read.

<source>
{article}
</source>
<summary>
{summary}
</summary>""",
    "coherence": """Score coherence from 1 to 5.
A score of 5 means the summary is logically organized and easy to follow.

<source>
{article}
</source>
<summary>
{summary}
</summary>""",
    "conciseness": """Score conciseness from 1 to 5.
A score of 5 means every sentence contributes important information and the
summary contains no unnecessary detail or repetition.

<source>
{article}
</source>
<summary>
{summary}
</summary>""",
    "consistency": """Score factual consistency from 1 to 5.
A score of 5 means every claim in the summary is supported by the source.
Penalize contradictions, fabricated details, and unsupported implications.

<source>
{article}
</source>
<summary>
{summary}
</summary>""",
    "relevance": """Score relevance from 1 to 5.
A score of 5 means the summary retains the source's central point and the most
important supporting details.

<source>
{article}
</source>
<summary>
{summary}
</summary>""",
}

evaluated_df = fresh_ai_view(summary_df)
for metric_name, prompt in EVAL_METRICS.items():
    evaluated_df = add_judge_metric(evaluated_df, metric_name, prompt)
evaluated_df = materialize(evaluated_df)
display(evaluated_df.select("summary", *EVAL_METRICS.keys()))


## 5. Results


In [ ]:
METRICS = ['fluency', 'coherence', 'conciseness', 'consistency', 'relevance']
results_pd = evaluated_df.select('sample_id', 'article', 'summary', 'fluency', 'coherence', 'conciseness', 'consistency', 'relevance').toPandas()

score_summary = pd.DataFrame({
    "Metric": ['Fluency', 'Coherence', 'Conciseness', 'Consistency', 'Relevance'],
    "Average score": [results_pd[metric].mean() for metric in METRICS],
    "Scored rows": [results_pd[metric].notna().sum() for metric in METRICS],
})
def quality_status(score, is_complete):
    if not is_complete or pd.isna(score):
        return "INCOMPLETE"
    return "PASS" if score >= 4 else "REVIEW" if score >= 3.5 else "FAIL"

score_summary["Status"] = [
    quality_status(score, scored_rows == len(results_pd))
    for score, scored_rows in zip(
        score_summary["Average score"],
        score_summary["Scored rows"],
    )
]
display(score_summary.round(2))

labels = score_summary["Metric"].tolist()
values = score_summary["Average score"].tolist()
fig = plt.figure(figsize=(13, 4.5))
bar_ax = fig.add_subplot(1, 2, 1)
bars = bar_ax.bar(labels, values, color="#0077aa")
bar_ax.set_ylim(0, 5)
bar_ax.set_ylabel("Score (1-5)")
bar_ax.set_title('Summary Quality')
bar_ax.axhline(y=4, color="#999999", linestyle="--", alpha=0.6)
bar_ax.tick_params(axis="x", rotation=20)
bar_ax.bar_label(bars, fmt="%.2f", padding=2)

if len(METRICS) >= 3:
    detail_ax = fig.add_subplot(1, 2, 2, polar=True)
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    detail_ax.plot(
        angles + angles[:1],
        values + values[:1],
        "o-",
        linewidth=2,
        color="#9955bb",
    )
    detail_ax.fill(
        angles + angles[:1],
        values + values[:1],
        alpha=0.25,
        color="#9955bb",
    )
    detail_ax.set_xticks(angles)
    detail_ax.set_xticklabels(labels)
    detail_ax.set_ylim(0, 5)
    detail_ax.set_title("Quality Profile", pad=20)
else:
    detail_ax = fig.add_subplot(1, 2, 2)
    detail_ax.hist(
        [results_pd[metric].dropna() for metric in METRICS],
        bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
        label=labels,
        alpha=0.7,
    )
    detail_ax.set_xticks([1, 2, 3, 4, 5])
    detail_ax.set_xlabel("Score")
    detail_ax.set_ylabel("Rows")
    detail_ax.set_title("Score Distribution")
    detail_ax.legend()
plt.tight_layout()
plt.show()

results_pd["scored_metrics"] = results_pd[METRICS].notna().sum(axis=1)
complete_rows = results_pd["scored_metrics"].eq(len(METRICS))
results_pd["average_score"] = (
    results_pd[METRICS].mean(axis=1).where(complete_rows).round(2)
)
results_pd["status"] = [
    quality_status(score, is_complete)
    for score, is_complete in zip(
        results_pd["average_score"],
        complete_rows,
    )
]


In [ ]:
breakdown_pd = results_pd[
    ["summary", *METRICS, "scored_metrics", "average_score", "status"]
].copy()
breakdown_pd["summary"] = breakdown_pd["summary"].str[:140] + "..."
display(breakdown_pd)


## 6. Optional: Summarize Multi-Column Rows

When `input_col` is omitted, `ai.summarize` synthesizes the full Spark row.
This is useful for support tickets and other structured records.


In [ ]:
TICKET_COLUMNS = ['ticket_id', 'customer', 'issue', 'priority', 'resolution']
ticket_rows = [('TKT-4021',
  'Sarah Martinez',
  'After our company migrated to the Enterprise plan last week, approximately 60 of our 200 users are unable '
  'to access the analytics dashboard. They see a spinner that loads indefinitely after login. The problem '
  "only affects users who were on the legacy 'Viewer' role - admin and editor users are fine. We've "
  "confirmed it's not a browser or network issue; affected users experience the same behavior on Chrome, "
  'Firefox, and Edge, and from both office and home networks. Our Q1 board meeting is next Tuesday and the '
  'CFO needs these dashboards.',
  'High',
  "Root cause identified: the Enterprise migration script did not map legacy 'Viewer' permissions to the new "
  "RBAC system. Ran a backfill script to assign the 'Dashboard Reader' role to all affected users. Verified "
  'access restored for all 60 users. Added a pre-migration validation check to prevent recurrence.'),
 ('TKT-4022',
  'James Chen',
  'Our automated billing pipeline has been double-charging a subset of customers since the January 15th '
  "platform update. We've identified 340 affected accounts totaling $47,200 in duplicate charges. The issue "
  'appears to be a race condition in the webhook handler - when a payment confirmation arrives within 200ms '
  'of the initial charge request, the system processes it as a new transaction instead of an acknowledgment. '
  'We need the duplicate charges reversed and a fix deployed before the next billing cycle on February 1st.',
  'Critical',
  'Deployed hotfix v2.14.3 that adds idempotency keys to the webhook handler, preventing duplicate '
  'processing. Initiated batch refund for all 340 affected accounts. Refunds will appear within 3-5 business '
  'days. Sent personalized apology emails to each affected customer with a 10% credit on their next '
  'invoice.'),
 ('TKT-4023',
  'Emily Rodriguez',
  'Three of our production ML models hosted on your platform started returning significantly degraded '
  "predictions after the v3.8.2 runtime update on January 20th. Our fraud detection model's precision "
  'dropped from 94.2% to 67.8%, causing a spike in false positives that blocked 1,200 legitimate '
  "transactions over the weekend. We've traced the issue to a change in how the runtime handles NumPy "
  'float32 precision - our model weights are being silently upcast to float64, which changes the inference '
  'behavior. Rolling back to v3.8.1 resolves the issue but we lose access to the new batch inference API '
  "we've already integrated.",
  'Critical',
  'Engineering confirmed the float32-to-float64 upcast bug in v3.8.2 runtime. Released v3.8.3 patch that '
  'preserves original dtype during model loading. Customer verified fraud model precision returned to 94.1% '
  'after patch. Filed internal incident report - adding dtype preservation tests to the CI pipeline to '
  'prevent regression.')]
tickets_df = spark.createDataFrame(ticket_rows, TICKET_COLUMNS)

ticket_summary_df = materialize(
    tickets_df.ai.summarize(
        output_col="ticket_summary",
        error_col="ticket_summary_error",
        **EXECUTOR_OPTIONS,
    )
)
display(ticket_summary_df.select("ticket_id", "priority", "ticket_summary"))


## 7. Optional Refinement: Concise Instructions

Use `instructions` for exact length and focus constraints while leaving
shared and per-call verbosity unset for the comparison.


In [ ]:
concise_df = materialize(
    df.ai.summarize(
        input_col="article",
        output_col="concise_summary",
        error_col="concise_executor_error",
        instructions=(
            "Summarize in one or two sentences. Keep only the single most important "
            "takeaway and essential supporting facts."
        ),
        **EXECUTOR_OPTIONS,
    )
)

concise_eval_df = add_judge_metric(
    fresh_ai_view(concise_df),
    "conciseness",
    EVAL_METRICS["conciseness"].replace("{summary}", "{concise_summary}"),
    column_prefix="concise_",
)
concise_eval_df = materialize(concise_eval_df)
display(concise_eval_df.select("article", "concise_summary", "concise_conciseness"))


In [ ]:
comparison_rows_pd = (
    evaluated_df.select("sample_id", "summary", "conciseness")
    .join(
        concise_eval_df.select(
            "sample_id", "concise_summary", "concise_conciseness"
        ),
        on="sample_id",
        how="full",
    )
    .toPandas()
)
required_columns = [
    "summary",
    "conciseness",
    "concise_summary",
    "concise_conciseness",
]
paired_mask = comparison_rows_pd[required_columns].notna().all(axis=1)
paired_count = int(paired_mask.sum())
excluded_count = int((~paired_mask).sum())
print(f"Paired rows: {paired_count} | Excluded rows: {excluded_count}")
if excluded_count:
    display(
        comparison_rows_pd.loc[
            ~paired_mask,
            ["sample_id", *required_columns],
        ]
    )
if not paired_count:
    raise ValueError("No rows have complete default and concise summary results.")

paired_pd = comparison_rows_pd.loc[paired_mask].copy()
paired_pd["default_words"] = paired_pd["summary"].str.split().str.len()
paired_pd["concise_words"] = paired_pd["concise_summary"].str.split().str.len()
display(
    paired_pd[
        [
            "sample_id",
            "summary",
            "default_words",
            "conciseness",
            "concise_summary",
            "concise_words",
            "concise_conciseness",
        ]
    ]
)

concise_summary_pd = pd.DataFrame({
    "Variant": ["Default", "With instructions"],
    "Average conciseness": [
        paired_pd["conciseness"].mean(),
        paired_pd["concise_conciseness"].mean(),
    ],
    "Average words": [
        paired_pd["default_words"].mean(),
        paired_pd["concise_words"].mean(),
    ],
})
display(concise_summary_pd.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
score_bars = axes[0].bar(
    concise_summary_pd["Variant"],
    concise_summary_pd["Average conciseness"],
    color=["#0077aa", "#22cc77"],
)
axes[0].set_ylim(0, 5)
axes[0].set_ylabel("Score (1-5)")
axes[0].set_title("Conciseness Comparison")
axes[0].bar_label(score_bars, fmt="%.2f", padding=2)

word_bars = axes[1].bar(
    concise_summary_pd["Variant"],
    concise_summary_pd["Average words"],
    color=["#0077aa", "#22cc77"],
)
axes[1].set_ylabel("Words")
axes[1].set_title("Average Summary Length")
axes[1].bar_label(word_bars, fmt="%.1f", padding=2)
plt.tight_layout()
plt.show()


## Interpreting Results

| Average score | Suggested action |
|---------------|------------------|
| **4.5-5.0** | Strong candidate for production validation |
| **4.0-4.4** | Good; inspect the lowest-scoring samples |
| **3.5-3.9** | Acceptable for iteration; refine data, prompts, or labels |
| **Below 3.5** | Investigate before broader use |
| **INCOMPLETE** | One or more judge scores are missing; inspect AI Function errors |

| Metric or issue | Likely cause | Next step |
|-----------------|--------------|-----------|
| Fluency | Awkward or noisy source text | Clean inputs or add style instructions |
| Conciseness | The output contains excess detail | Add explicit length instructions |
| Consistency | Unsupported claims | Review flagged summaries against the source |
| Relevance | Key information is missing | Specify the required focus |

### Improving Quality

- Use `instructions` for exact constraints on length, audience, tone, and focus.
- Use a targeted per-call `verbosity` override when broad response detail should
  change; leave the shared default unset.
- Treat consistency failures as high priority and inspect them against the source.
- For multi-column rows, omit `input_col` to summarize the full row.

```python
low_detail_df = df.ai.summarize(
    input_col="article",
    output_col="low_detail_summary",
    error_col="low_detail_error",
    verbosity="low",
    **EXECUTOR_OPTIONS,
)
```

Keep the judge configuration fixed for comparisons, and confirm release decisions with representative human-reviewed samples.

## Learn More

- [ai.summarize PySpark documentation](https://learn.microsoft.com/fabric/data-science/ai-functions/pyspark/summarize)
- [AI Functions overview](https://learn.microsoft.com/fabric/data-science/ai-functions/overview)
